In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import ipywidgets as widgets
from IPython.display import display

In [4]:
default = "Seleccionar una opcion"
files = [f for f in os.listdir('F:/') if f.endswith('.csv')]
files.insert(0, default)
file_selector = widgets.Dropdown(options=files, description='CSV File:', default=default)
display(file_selector)

def plot_csv(change):
    if change['type'] == 'change' and change['name'] == 'value' and change['new'] != default:
        df = pd.read_csv(change['new'], sep=";")
        display(df.head())
        fig = make_subplots(rows=1, cols=3, subplot_titles=["Current", "Resistance", "Error"])
        fig.add_trace(go.Scatter(x=df.index, y=df["Current"], name="Current"), row=1, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df["Resistance"], name="Resistance"), row=1, col=2)
        fig.add_trace(go.Scatter(x=df.index, y=df["Error"], name="Error"), row=1, col=3)
        fig.show()

file_selector.observe(plot_csv)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'F:/'

In [9]:
from pandas.errors import EmptyDataError
archivo_salida = "datos.csv"
columns = ["Voltage", "Current", "PWM Value", "Error", "Integral", "Derivative", "R_Target"]
try:
    df = pd.read_csv(archivo_salida)
except EmptyDataError:
    print(f"El archivo {archivo_salida} está vacío o no existe.")
    df = pd.DataFrame()
df.columns = columns
df["error%"] = (df["Error"] / (1000 * df["Voltage"] / df["R_Target"])) * 100
df["Resistance"] = 1000* df["Voltage"] / df["Current"]
df["R_error%"] = (df["Resistance"] - df["R_Target"]) / df["R_Target"] * 100

fig = make_subplots(rows=3, cols=2)
fig.add_trace(go.Scatter(x=df.index, y=df["Current"], name="Current"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["Error"], name="Error"), row=1, col=2)
fig.add_trace(go.Scatter(x=df.index, y=df["Derivative"], name="Derivative"), row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["Integral"], name="Integral"), row=2, col=2)
fig.add_trace(go.Scatter(x=df.index, y=df["R_Target"], name="R_Target"), row=3, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["R_error%"], name="R_error%"), row=3, col=2)

fig.update_layout(
    title="Carga Electrónica - Analisis de rendimiento",
    legend_title="Datos",
    template="plotly_dark",
)
# fig.add_trace(go.Scatter(x=df.index, y=df["error%"], name="error%"), row=3, col=2)